# Robust galaxy morphology across environment-selected samples

This notebook turns the action items from the 13 July 2026 research meeting into a reproducible analysis. It compares robust early-type and late-type galaxy classifications in the `highlum` and `highdens` cross-match products.

The notebook is intentionally explanatory: every figure is preceded by its question and construction, then followed by guidance on interpretation and limitations. Generated outputs are local artifacts and are not committed.

In [ ]:
import json
import os
import platform
from pathlib import Path

import numpy as np

SEED = 20260713
MAX_SEPARATION_ARCSEC = 1.0
FLUX_RADIUS_CUT = 50.0
PLOT_SAMPLE_SIZE = 100_000

def find_project_root() -> Path:
    configured = os.environ.get('GALAXY_PROJECT_ROOT')
    candidates = [Path(configured)] if configured else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'data' / 'README.md').is_file():
            return candidate.resolve()
    raise FileNotFoundError('project root requires requirements.txt and data/README.md')

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'meeting-2026-07-13'


## 1. Scientific context and questions

Modern surveys contain far more galaxy images than a research team can inspect consistently by hand. The Vega-Ferrero catalogue applies ensembles of convolutional neural networks to DES images and reports probabilities and flags for two related questions: whether a galaxy has late-type morphology and whether a disk is viewed edge-on.

In this notebook, **early-type galaxy (ETG)** and **late-type galaxy (LTG)** are operational catalogue classes, not claims that every galaxy belongs to a perfect physical binary. We ask: (1) how many robust ETGs and LTGs occur in each cross-match, (2) how their classification probabilities and observable size/brightness measures differ, and (3) whether patterns differ between the files called `highlum` and `highdens`.

The environmental catalogues have their own selection functions. A different ETG/LTG fraction can therefore arise from population differences, survey limits, matching, or classification uncertainty. The analysis describes associations; it does not establish environmental causation.

## 2. Data inventory and provenance

Five local FITS files have distinct roles. `DES_DR1_CNN_morphological_catalog.fit` is the 26.97-million-row parent morphology catalogue. The two `redspell_*` files are environmental source catalogues. The two `match_VF_*` files are derived coordinate matches that append Vega-Ferrero morphology fields and a match separation to the environmental records.

Only the parent morphology catalogue and the two match products are analyzed directly here. Source catalogues are inspected for provenance. Raw inputs, derived match products, and generated results remain in separate directories so an analysis cannot silently overwrite its evidence.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from astropy.table import Table
from src.galaxy_analysis.catalog import inspect_catalog, random_indices, read_columns

CATALOG_PATHS = {
    'parent_morphology': PROJECT_ROOT / 'data' / 'DES_DR1_CNN_morphological_catalog.fit',
    'highlum_source': PROJECT_ROOT / 'data' / 'raw' / 'red-sequence' / 'redspell_highlum7_final.fits',
    'highdens_source': PROJECT_ROOT / 'data' / 'raw' / 'red-sequence' / 'redspell_highdens7_final.fits',
    'highlum': PROJECT_ROOT / 'data' / 'processed' / 'crossmatches' / 'vega-ferrero' / 'match_VF_highlum.fits',
    'highdens': PROJECT_ROOT / 'data' / 'processed' / 'crossmatches' / 'vega-ferrero' / 'match_VF_highdens.fits',
}
schemas = {name: inspect_catalog(path) for name, path in CATALOG_PATHS.items()}
inventory = Table(rows=[
    (name, str(schema.path.relative_to(PROJECT_ROOT)), schema.row_count, schema.column_count, schema.hdu_index, schema.extname)
    for name, schema in schemas.items()
], names=('catalog', 'path', 'rows', 'columns', 'table_hdu', 'extname'))
inventory


## 3. Reproducibility configuration

Every random operation uses seed `20260713`, the meeting date. Coordinate matches are accepted only through 1.0 arcsec. The proposed `FLUX_RADIUS_R < 50` cut is used only for an explicitly labeled diagnostic figure until its meaning is confirmed. Scatter rendering is capped at 100,000 deterministic points, while numerical statistics use all valid rows.

The notebook writes tables, figures, and run metadata below `outputs/meeting-2026-07-13/`. That directory is ignored by Git: generated outputs can always be rebuilt and must not be confused with source code or input data.

## 4. FITS schema and quality validation

A catalogue can be opened successfully and still be unsuitable for analysis. We therefore check structure, domains, missing values, duplicate identifiers, and coordinate-match quality before calculating any scientific summary. The match tolerance is an **angular separation** on the sky. One arcsecond is not a physical distance: its physical scale depends on redshift and cosmology.

`Separation <= 1 arcsec` follows the Topcat procedure demonstrated in the meeting. Invalid rows are counted rather than silently discarded. Missing coordinate units in a FITS header are reported as metadata limitations even when the catalogue convention indicates degrees.

In [ ]:
from dataclasses import asdict

from src.galaxy_analysis.selection import audit_filter, robust_masks, valid_value_mask, validate_separation

CORE_COLUMNS = (
    'COADD_OBJECT_ID', 'object_id', 'RA_2', 'DEC_2', 'MAG_AUTO_R',
    'FLUX_RADIUS_R', 'MP_LTG', 'MP_EdgeOn', 'FLAG_LTG', 'FLAG_EdgeOn',
    'Separation', 'P1_LTG', 'P2_LTG', 'P3_LTG', 'P4_LTG', 'P5_LTG',
    'P1_EdgeOn', 'P2_EdgeOn', 'P3_EdgeOn', 'P4_EdgeOn', 'P5_EdgeOn',
)
highlum = read_columns(CATALOG_PATHS['highlum'], CORE_COLUMNS)
quality_masks = {
    'RA_2 in [0, 360) deg': valid_value_mask(highlum['RA_2'], 'ra_deg'),
    'DEC_2 in [-90, 90] deg': valid_value_mask(highlum['DEC_2'], 'dec_deg'),
    'FLUX_RADIUS_R positive': valid_value_mask(highlum['FLUX_RADIUS_R'], 'positive'),
    'MP_LTG in [0, 1]': valid_value_mask(highlum['MP_LTG'], 'probability'),
    'MP_EdgeOn in [0, 1]': valid_value_mask(highlum['MP_EdgeOn'], 'probability'),
    'Separation in [0, 1] arcsec': valid_value_mask(highlum['Separation'], 'nonnegative') & (highlum['Separation'] <= MAX_SEPARATION_ARCSEC),
}
audit_rows = [audit_filter('highlum quality', rule, mask) for rule, mask in quality_masks.items()]
separation_check = validate_separation(highlum['Separation'], MAX_SEPARATION_ARCSEC)
duplicate_morphology_ids = len(highlum['COADD_OBJECT_ID']) - len(np.unique(highlum['COADD_OBJECT_ID']))
duplicate_environment_ids = len(highlum['object_id']) - len(np.unique(highlum['object_id']))
assert separation_check.is_valid, f'{separation_check.invalid_count} matches exceed quality limits'
audit_table = Table(rows=[tuple(asdict(row).values()) for row in audit_rows], names=tuple(asdict(audit_rows[0])))
audit_table


In [ ]:
highlum_output = OUTPUT_ROOT / 'highlum'
highlum_tables = highlum_output / 'tables'
highlum_figures = highlum_output / 'figures'
highlum_tables.mkdir(parents=True, exist_ok=True)
highlum_figures.mkdir(parents=True, exist_ok=True)
quality_rows = []
for name, values in highlum.items():
    array = np.asarray(values)
    if np.issubdtype(array.dtype, np.number):
        numeric = array.astype(float)
        finite = np.isfinite(numeric)
        valid = numeric[finite]
        stats = (float(valid.min()), float(valid.max()), float(valid.mean()), float(np.median(valid))) if valid.size else (None, None, None, None)
        n_finite, n_missing = int(finite.sum()), int((~finite).sum())
    else:
        stats = (None, None, None, None)
        n_finite, n_missing = int(array.size), 0
    quality_rows.append(('highlum', name, str(array.dtype), schemas['highlum'].column_units.get(name), len(array), n_finite, n_missing, 0, *stats))
quality_table = Table(rows=quality_rows, names=('catalog', 'column', 'dtype', 'unit', 'n_total', 'n_finite', 'n_missing', 'n_out_of_range', 'min', 'max', 'mean', 'median'))
quality_table.write(highlum_tables / 'catalog_quality.csv', format='ascii.csv', overwrite=True)
quality_report = {
    'catalog': 'highlum',
    'rows': schemas['highlum'].row_count,
    'columns': schemas['highlum'].column_count,
    'separation_unit': schemas['highlum'].column_units['Separation'],
    'separation_invalid_count': separation_check.invalid_count,
    'separation_maximum_valid_arcsec': separation_check.maximum_valid,
    'duplicate_morphology_ids': int(duplicate_morphology_ids),
    'duplicate_environment_ids': int(duplicate_environment_ids),
    'warnings': ['RA/DEC units are not declared in the FITS header; catalogue convention is degrees'],
}
(highlum_output / 'quality_report.json').write_text(json.dumps(quality_report, indent=2), encoding='utf-8')
audit_table.write(highlum_tables / 'filter_audit.csv', format='ascii.csv', overwrite=True)
quality_report


## 5. Robust ETG and LTG selection

The morphology flag encodes both class and confidence tier. The primary analysis uses exactly `FLAG_LTG == 4` for robust ETGs and `FLAG_LTG == 5` for robust LTGs. Flags 0–3 are excluded from the robust comparison. Treating every even flag as ETG and every odd flag as LTG would mix lower-confidence objects into the samples and would not follow the meeting decision.

A small robust-LTG fraction is not, by itself, evidence that the classifier failed. The environmental source catalogue may preferentially select luminous, dense-region, or red-sequence systems, producing **sample-selection bias** toward ETGs. Class imbalance is therefore reported as a result and interpreted cautiously.

In [ ]:
highlum_masks = robust_masks(highlum['FLAG_LTG'])
flag_values, flag_counts = np.unique(highlum['FLAG_LTG'], return_counts=True)
highlum_flag_counts = dict(zip(flag_values.astype(int), flag_counts.astype(int)))
assert len(highlum['FLAG_LTG']) == 34_768
assert highlum_flag_counts == {0: 5220, 1: 3039, 2: 344, 3: 404, 4: 24871, 5: 890}
assert int(highlum_masks.robust_etg.sum()) == 24_871
assert int(highlum_masks.robust_ltg.sum()) == 890
selection_table = Table(
    rows=[('robust ETG', 4, int(highlum_masks.robust_etg.sum())), ('robust LTG', 5, int(highlum_masks.robust_ltg.sum()))],
    names=('class', 'FLAG_LTG', 'count'),
)
selection_table['fraction_of_catalog'] = selection_table['count'] / len(highlum['FLAG_LTG'])
selection_table


## 6. Classification probabilities and model agreement

A `FLAG_LTG` value is a catalogue decision assembled from model outputs and confidence rules. `MP_LTG` and `MP_EdgeOn` are continuous aggregate probabilities, while P1–P5 preserve the five model predictions. These quantities answer different questions: the flag defines the selected class; the aggregate probability shows the strength of the model output; and the spread across P1–P5 shows inter-model disagreement.

We report mean, median, sample standard deviation (`ddof=1`), percentiles, and interquartile range. Thresholds 0.5, 0.6, and 0.8 are sensitivity checks only. Agreement with flag 5 does not prove correctness because the flag is not an independent human-labelled ground truth. High confidence can still be systematically wrong.

In [ ]:
from src.galaxy_analysis.statistics import describe_values, flag_count_rows, model_dispersion, threshold_rows

highlum_flag_rows = flag_count_rows('highlum', highlum['FLAG_LTG'])
flag_table = Table(rows=[tuple(asdict(row).values()) for row in highlum_flag_rows], names=tuple(asdict(highlum_flag_rows[0])))
summary_rows = []
for class_name, mask in [('all_valid', np.ones(len(highlum['FLAG_LTG']), dtype=bool)), ('robust_etg', highlum_masks.robust_etg), ('robust_ltg', highlum_masks.robust_ltg)]:
    for variable in ('MP_LTG', 'MP_EdgeOn', 'MAG_AUTO_R', 'FLUX_RADIUS_R', 'Separation'):
        summary_rows.append(describe_values('highlum', class_name, variable, highlum[variable][mask]))
summary_table = Table(rows=[tuple(asdict(row).values()) for row in summary_rows], names=tuple(asdict(summary_rows[0])))
summary_table.rename_columns(['minimum', 'maximum'], ['min', 'max'])
threshold_result_rows = threshold_rows('highlum', highlum['MP_LTG'], highlum['FLAG_LTG'])
threshold_table = Table(rows=[tuple(asdict(row).values()) for row in threshold_result_rows], names=tuple(asdict(threshold_result_rows[0])))
ltg_models = np.column_stack([highlum[f'P{i}_LTG'] for i in range(1, 6)])
edgeon_models = np.column_stack([highlum[f'P{i}_EdgeOn'] for i in range(1, 6)])
ltg_dispersion = model_dispersion(ltg_models)
edgeon_dispersion = model_dispersion(edgeon_models)

baseline = {
    ('robust_etg', 'MP_LTG'): (0.01198184, 0.00148256, 0.02721692),
    ('robust_etg', 'MP_EdgeOn'): (0.02750497, 0.00760955, 0.07180363),
    ('robust_ltg', 'MP_LTG'): (0.95182968, 0.96886268, 0.04962162),
    ('robust_ltg', 'MP_EdgeOn'): (0.13879124, 0.06322150, 0.18570282),
}
for row in summary_rows:
    key = (row.class_name, row.variable)
    if key in baseline:
        np.testing.assert_allclose((row.mean, row.median, row.std_ddof1), baseline[key], atol=1e-7, rtol=0)

flag_table.write(highlum_tables / 'flag_counts.csv', format='ascii.csv', overwrite=True)
summary_table.write(highlum_tables / 'summary_statistics.csv', format='ascii.csv', overwrite=True)
threshold_table.write(highlum_tables / 'threshold_sensitivity.csv', format='ascii.csv', overwrite=True)
display(flag_table)
display(summary_table[["class_name", "variable", "n_valid", "mean", "median", "std_ddof1", "p25", "p75"]])
display(threshold_table)


### Reading the probability summaries

Robust ETGs should have `MP_LTG` concentrated near zero, while robust LTGs should concentrate near one; the verified medians reflect exactly that operational definition. `MP_EdgeOn` is not another morphology class: it represents viewing orientation and is especially relevant for disks whose spiral structure may be hidden in projection.

The P1–P5 standard deviation and range measure disagreement among the five trained networks. They are useful uncertainty diagnostics, but they do not include every source of uncertainty—such as image depth, selection effects, domain shift, or incorrect labels used during training.

## 7. Magnitude–size relation

## 8. Probability, faintness, and edge-on orientation

## 9. Sky distribution

## 10. High-luminosity versus high-density comparison

## 11. Random extraction from the parent morphology catalogue

## 12. Conclusions, limitations, and questions for the advisor